# 01. Среда и сохранённая карта конфигураций

Основа — утверждённая TASK R1.4, §§2–6. Эта тетрадь **не запускает** новую серию и не использует старые CSV. Работающее событийное ядро и ручной пример доступны в `00_Старт.ipynb`. Здесь читаются только сохранённые результаты с проверкой цепочек всех отображаемых клеток.

Критерий: **p95 ≤ 200 мс И потери ≤ 0,1%**. p95 относится к завершённым сообщениям; потери — ко всем поступлениям. Пилот, итоговые seed и независимое подтверждение — разные этапы.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
root = Path.cwd()
while root != root.parent and not (root/'efc/companion.py').exists():
    root = root.parent
if not (root/'efc/companion.py').exists():
    try:
        import google.colab
    except ImportError:
        raise RuntimeError('Запустите из копии night/fefc-v3 или в Colab.')
    root = Path('/content/fefc-r14-companions')
    if not root.exists():
        subprocess.run(['git','clone','--single-branch','--branch','night/fefc-v3',
            'https://github.com/annfridreksen/Forecast_Edge_Fog_Cloud.git',str(root)],check=True)
    if not (root/'efc/companion.py').exists():
        raise RuntimeError('Копия устарела: companion.py отсутствует. Молчаливого обновления нет.')
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(root/'requirements.txt')],check=True)
os.chdir(root)
sys.path.insert(0,str(root))
SOURCE_SHA = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip() if (root/'.git').exists() else os.environ.get('FEFC_SOURCE_SHA','архив: SHA в реестре')
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib','inline')
import matplotlib.pyplot as plt
import pandas as pd
from efc import companion as c
print('Источник:',SOURCE_SHA,'; Python:',sys.version.split()[0])
print('Только чтение и схемы: имитация, обучение, прогноз и применение плана не запускаются.')

## 1. Загрузка, а не новый расчёт

Укажите `FEFC_CHECKPOINT` — каталог распакованного накопленного архива конкретного Actions; по умолчанию `results/r14/night-map`. При отсутствии или повреждении архива научные числа отсутствуют, не равны нулю. Проверка хешей и классификации не повторяет bootstrap и не подтверждает точность его покрытия.

In [ ]:
checkpoint = Path(os.environ.get('FEFC_CHECKPOINT','results/r14/night-map'))
snapshot = c.overview(checkpoint)
print('Источник данных:',checkpoint,'; состояние:',snapshot['state'])
print(snapshot['note'])
print('Статусы проверенных строк:',snapshot['counts'])
if snapshot['rows']:
    print('Хеш manifest:',snapshot['manifest_sha256'])
    print('Хеш map (канонический JSON):',snapshot['map_sha256'])
    display(pd.DataFrame([{k:r.get(k) for k in ('key','arrival_rate_s','n_fog','buffer','status','main_seeds_completed')} for r in snapshot['rows']]))

## 2. Одинаковый внешний поток

Выбираем одну интенсивность λ и сравниваем число Fog и буфер B. Изменение числа Fog **не меняет внешний поток**. B — места ожидания, включая обслуживаемое сообщение вместимость равна B+1. Цвета различают допустимость, недопустимость, недостаточность данных и отсутствие расчёта. Статистическая минимальность не заявляется. Кандидатов фиксирует контроллер, не эта тетрадь.

In [ ]:
from efc.map_view import configuration_map_figure
if snapshot['rows']:
    selected_rate = min(r['arrival_rate_s'] for r in snapshot['rows'])
    fig = configuration_map_figure(snapshot,selected_rate)
    display(fig); plt.close(fig)
else:
    print('Карта не рисуется: проверенных строк нет.')

## 3. p95 и потери от нагрузки

Ниже фиксированы Fog=50, B=5; ρ50=λ·0,055/50 — **предложенная**, а не измеренная нагрузка. Каждая точка требует итогового `main.json` и сохранённого 95%-го доверительного интервала (ДИ). Пропуски не соединяются. Здесь нет результатов пилота, синтетической замены недостающих точек или обучения.

In [ ]:
for metric in ('p95_ms','drop_fraction'):
    points = c.curve_points(snapshot,metric,50,5)
    if points: display(pd.DataFrame(points))
    fig = c.curve_figure(snapshot,metric,50,5)
    display(fig); plt.close(fig)

## 4. Границы вывода

`INSUFFICIENT_DATA` после предела прогрева не означает доказанную нестационарность или недопустимость. Нулевые наблюдённые потери не доказывают нулевой риск. Неполная карта не обосновывает минимум всей сетки. H(t), очередь и шесть составляющих короткого примера показаны отдельно в тетради 00; их нельзя подписывать как стационарные результаты.

| Утверждение | Ячейка | Проверяемый источник |
|---|---|---|
| Состояния сохранённой карты | `r14-01-load`, `r14-01-map` | map → main/pilot; manifest/config_id |
| p95/потери и ДИ | `r14-01-curves` | `cells/<key>/main.json`, поле summary, SHA в таблице точек |
| Нет новых численных опытов | `r14-01-receipt` | режим чтения; адресные тесты запрещают вызов ядра |

Статья и полная карта этой тетрадью не объявляются завершёнными.

In [ ]:
print(json.dumps(dict(notebook='01',source=SOURCE_SHA,state=snapshot['state'],counts=snapshot['counts'],simulation_executed=False,forecast_executed=False,training_executed=False,colab_browser_checked=False),ensure_ascii=False,indent=2))